# Train the in-house decision model (Colab GPU)

Trains the Trading Agent's own decision model, following the TDD:
1. **Dataset**: years of Binance prices (the historical bootstrap), plus logged snapshots and closed trades if you have them.
2. **Step 1, SFT**: learn "context → {answer, confidence}".
3. **Step 2, RL (GRPO)**: reward real profit and honest confidence.
4. **Exam**: compare on the newest 20% of data against "always buy" and "always hold".
   The model is only worth using if it beats "always hold".

**Before running:** Runtime → Change runtime type → **T4 GPU** (free) or better. Then Runtime → Run all.
The default model (Qwen2.5-0.5B) trains in about 15–30 minutes on a T4. The 30–40B target needs a rented A100/H100
(set `BASE_MODEL` and `QLORA = True`).

In [ ]:
REPO = "A-n-singh/stock-trading-universe"
BRANCH = "claude/new-session-uimdbt"
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # try "Qwen/Qwen2.5-1.5B-Instruct" next; "Qwen/Qwen2.5-32B-Instruct" on a big GPU
QLORA = False                                # True for 30-40B models (4-bit)
SFT_STEPS, RL_STEPS = 300, 60                # raise for better models, lower for a quick try
SAVE_TO_DRIVE = True

import os, subprocess, sys
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, f"https://github.com/{REPO}.git", "/content/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
os.chdir("/content/repo")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[backtest,training]"] + (["bitsandbytes"] if QLORA else []), check=True)
sys.path.insert(0, "/content/repo/src")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to a GPU")

In [ ]:
OUT = "/content/drive/MyDrive/trading-universe" if SAVE_TO_DRIVE else "/content/work"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)
# If you have run the live system, copy its runs/ folder to OUT/runs so its snapshots and trades are included.
!python -m trading_universe.training build --coins BTCUSDT,ETHUSDT,SOLUSDT,BNBUSDT --out {OUT}/data --runs {OUT}/runs

## Step 1: supervised fine-tuning (SFT)

In [ ]:
flags = "--qlora --bf16" if QLORA else ""
!python -m trading_universe.training sft --train {OUT}/data/train.jsonl --base {BASE_MODEL} --out {OUT}/models/sft --batch 8 --max-steps {SFT_STEPS} {flags}

## Step 2: reinforcement learning (GRPO): profit + honest confidence

In [ ]:
!python -m trading_universe.training rl --train {OUT}/data/train.jsonl --base {OUT}/models/sft-merged --out {OUT}/models/rl --batch 8 --max-steps {RL_STEPS}

## Exam on the newest 20% of data

Profit is in R (1 R = one stop-loss hit = your ₹250 risk). **Use the model only if it beats `always-hold`.**

In [ ]:
!python -m trading_universe.training evaluate --model {OUT}/models/sft-merged --test {OUT}/data/test.jsonl
!python -m trading_universe.training evaluate --model {OUT}/models/rl-merged --test {OUT}/data/test.jsonl

## Using the model

- **Offline check:** the numbers above.
- **In the Trading Agent:** copy `models/rl-merged` next to your `runs/` folder and start the system with it
  (`HFDecisionModel`). It becomes a fourth check that must also say "buy".
- **Fast serving:** quantize and serve it (see `python -m trading_universe.training merge --help` output),
  then point `HTTPDecisionModel` at the server.
- **Retraining:** once there are about 1,000 new closed trades, run
  `python -m trading_universe.training retrain --runs runs --live models/rl-merged`. The new model replaces the
  live one only if it wins on recent trades it hasn't seen.